#### 1. 라이브러리 임포트

In [15]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas() 

# import re
# import contractions

import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.utils import resample
from tensorflow.keras import optimizers, callbacks
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, SimpleRNN, Dense)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#### 2. 시드 고정

In [16]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

#### 3. 데이터 로드

In [17]:
# parquet 형식 데이터 로드
df = pd.read_parquet('final_data.parquet')

# 데이터 크기 확인
print(f"데이터 크기: {df.shape}")

# 데이터 비율 확인
print(f"데이터 비율: {df['fake'].value_counts()}")

# 샘플 데이터 확인
df.sample(5)

데이터 크기: (70000, 7)
데이터 비율: fake
1    35000
0    35000
Name: count, dtype: int64


,review_text,fake,basic_linguistic_list,readability_list,sentiment_list,behavioral_list,clean_text
46730,My girlfriend and I were looking for a romanti...,1,"[122.0, 82.0, 7.0, 451.0, 350.0, 15.0, 58.0, 1...","[12.16174496147169, 67.09504065040652, 7.29609...","[0.4193181818181819, 0.7090909090909091, 0.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0]",my girlfriend and i were looking for a romanti...
48393,The food here is excellent. The mac and cheese...,1,"[66.0, 50.0, 6.0, 252.0, 197.0, 3.0, 37.0, 6.0...","[7.168621630094336, 86.70466666666667, 3.23600...","[0.6994444444444444, 0.7288888888888889, 0.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0]",the food here is excellent. the mac and cheese...
41416,I came here with a friend around 3pm. The plac...,1,"[155.0, 123.0, 10.0, 642.0, 502.0, 5.0, 97.0, ...","[7.168621630094336, 85.85950000000001, 4.38899...","[0.2580555555555555, 0.5577777777777778, 1.0, ...","[4.0, 208.0, 69.33333333333333, 32.34707611722...",i came here with a friend around 3pm. the plac...
34506,"pizza I""ll give a 4....food..eh,..3 1/2.....we...",1,"[59.0, 42.0, 1.0, 241.0, 163.0, 5.0, 30.0, 7.0...","[8.07648339933343, 67.03437500000001, 4.996250...","[-0.2499999999999999, 0.4333333333333333, 2.0,...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 0.0]",pizza i ll give a 4 food eh 3 1 2 went there 3...
43725,This place is awesome. For a single guy that d...,0,"[44.0, 31.0, 4.0, 163.0, 127.0, 3.0, 22.0, 8.0...","[8.07648339933343, 76.16229838709678, 4.561532...","[0.3948660714285714, 0.5785714285714285, 0.0, ...","[12.0, 359.0, 32.63636363636363, 48.4897364960...",this place is awesome. for a single guy that d...


#### 4. 텍스트 전처리

In [18]:
# def clean_text_keep_words(text, min_words=20):

#     text = str(text)

#     text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # URL 제거
#     text = re.sub(r"<[^>]+>", " ", text)                # HTML/XML 태그 제거
    
#     try:
#         text = contractions.fix(text)                   # 축약어 복원
#     except Exception:
#         return None                                     # contractions 처리 오류 행 제거
    
#     # text = re.sub(r"[^A-Za-z0-9\s]", " ", text)         # 특수문자 제거
#     text = re.sub(r"\.(?!\s|$)", " ", text)       # 문장 끝이 아닌 점 제거
#     text = re.sub(r"[^A-Za-z0-9\s.]", " ", text)  # 나머지 특수문자 제거
#     text = text.lower()                                 # 소문자 변환
#     text = re.sub(r"\s+", " ", text).strip()            # 공백 정리

#     if len(text.split()) < min_words:                   # 최소 단어 수 필터링
#         return None

#     return text

In [19]:
# # 데이터프레임 복사
# df_clean_text = df.copy()

# # 텍스트 정제 함수 적용
# df_clean_text["text_clean"] = df_clean_text["review_text"].astype(str).progress_apply(clean_text_keep_words)

# # None 값 제거
# df_clean_text = df_clean_text.dropna(subset=["text_clean"]).reset_index(drop=True)

# # 결과 확인
# print(f"텍스트 정제 전 데이터 수: {len(df):,}")
# print(f"텍스트 정제 후 데이터 수: {len(df_clean_text):,}")
# print(f"제거된 데이터 수: {len(df) - len(df_clean_text):,}")

# # 결과 샘플 확인
# df_clean_text[["review_text", "text_clean"]].sample(5)

#### 5. Train / Validation / Test 분할

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["fake"].values.astype("float32"), test_size=0.2, stratify=df["fake"], random_state=42
)

#### 6. Tokenize

In [29]:
MAX_WORDS = 10000
MAX_LEN = 128

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

train_seq = tokenizer.texts_to_sequences(X_train)
test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(test_seq,  maxlen=MAX_LEN, padding="post", truncating="post")

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

print(X_train_pad.shape, X_test_pad.shape)
print("vocab_size:", vocab_size)


(56000, 128) (14000, 128)
vocab_size: 10000


In [30]:
print(len(tokenizer.word_index) + 1)
print(min(MAX_WORDS, len(tokenizer.word_index) + 1))
print(f"OOV 비율: {sum(token == tokenizer.word_index.get('<OOV>') for seq in train_seq for token in seq) / sum(len(seq) for seq in train_seq):.4f}")

52486
10000
OOV 비율: 0.0189


#### 7. RNN 함수

In [31]:
def build_rnn():
    text_input = Input(shape=(MAX_LEN,))
    x = Embedding(input_dim=vocab_size, output_dim=128)(text_input)
    x = SimpleRNN(128, return_sequences=False)(x)
    
    x = Dense(2048, activation="relu")(x)
    x = Dense(1024, activation="relu")(x)
    x = Dense(512, activation="relu")(x)
    x = Dense(256, activation="relu")(x)
    x = Dense(128, activation="relu")(x)
    
    output = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=text_input, outputs=output)

    return model

#### 8. 모델 생성

In [32]:
model = build_rnn()

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 128, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 2048)           │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,364,417 (16.65 MB)

 Trainable params: 4,364,417 (16.65 MB)

 Non-trainable params: 0 (0.00 B)

#### 9. 모델 컴파일

In [33]:
model.compile(optimizer=optimizers.Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])

#### 10. 콜백 설정

In [34]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

#### 11. 모델 학습

In [35]:
history = model.fit(x=X_train_pad, y=y_train, validation_split=0.125, epochs=20, batch_size=32, callbacks=callbacks, verbose=1)

Epoch 1/20


I0000 00:00:1780513861.746558   47112 dot_merger.cc:481] Merging Dots in computation: functional_2_1_simple_rnn_2_1_while_body_123104_grad_123387_const_0__.22.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1780513861.746613   47112 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_124209__.25


1527/1532 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5835 - loss: 0.6771

I0000 00:00:1780513872.651212   47111 dot_merger.cc:481] Merging Dots in computation: functional_2_1_simple_rnn_2_1_while_body_123104_grad_123387_const_0__.22.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1780513872.651265   47111 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_124209__.25


1532/1532 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.5943 - loss: 0.6709 - val_accuracy: 0.5574 - val_loss: 0.6817
Epoch 2/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.6371 - loss: 0.6405 - val_accuracy: 0.6360 - val_loss: 0.6509
Epoch 3/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.6785 - loss: 0.6016 - val_accuracy: 0.6190 - val_loss: 0.6679
Epoch 4/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.7088 - loss: 0.5459 - val_accuracy: 0.6221 - val_loss: 0.7521
Epoch 5/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.7605 - loss: 0.4726 - val_accuracy: 0.6329 - val_loss: 1.0195
Epoch 6/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.7907 - loss: 0.4239 - val_accuracy: 0.6274 - val_loss: 1.2595
Epoch 7/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8149 - loss: 0.3797 - val_accuracy: 0.6267 - val_loss: 1.3324


#### 12. 예측 및 성능 계산

In [36]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int)

print(f"acc  : {accuracy_score(y_test, y_pred):.4f}")
print(f"prec : {precision_score(y_test, y_pred):.4f}")
print(f"rec  : {recall_score(y_test, y_pred):.4f}")
print(f"f1   : {f1_score(y_test, y_pred):.4f}")

438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
acc  : 0.6409
prec : 0.6639
rec  : 0.5706
f1   : 0.6137
